In [ ]:
!git clone https://github.com/bigohofone/outfit-transformer.git
%cd outfit-transformer

Cloning into 'outfit-transformer'...
remote: Enumerating objects: 1094, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 1094 (delta 162), reused 134 (delta 134), pack-reused 888 (from 2)
Receiving objects: 100% (1094/1094), 182.41 MiB | 22.52 MiB/s, done.
Resolving deltas: 100% (545/545), done.
/content/outfit-transformer


In [ ]:
!conda install --file environment.yml  # won't work in Colab
# Instead use pip:
!pip install torch torchvision
!pip install transformers
!pip install faiss-cpu  # or faiss-cpu if no GPU
!pip install gdown
!pip install wandb  # only needed if you want to train

/bin/bash: line 1: conda: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.2 MB/s eta 0:00:00


In [ ]:
!mkdir -p checkpoints
!gdown --id 1mzNqGBmd8UjVJjKwVa5GdGYHKutZKSSi -O checkpoints.zip
!unzip checkpoints.zip -d ./checkpoints
!rm checkpoints.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1mzNqGBmd8UjVJjKwVa5GdGYHKutZKSSi
From (redirected): https://drive.google.com/uc?id=1mzNqGBmd8UjVJjKwVa5GdGYHKutZKSSi&confirm=t&uuid=2553756c-27f1-4138-b5cf-0c504f4ab169
To: /content/outfit-transformer/checkpoints.zip
100% 1.15G/1.15G [00:16<00:00, 69.9MB/s]
Archive:  checkpoints.zip
  inflating: ./checkpoints/compatibillity_clip_best.pth  
  inflating: ./checkpoints/complementary_clip_best.pth  


In [ ]:
!ls checkpoints/

compatibillity_clip_best.pth  complementary_clip_best.pth


In [ ]:
!ls src/                 # to see what's there
!ls src/models/
!ls src/data/
!ls src/demo/
!ls src/evaluation/
!ls src/run/
!ls src/utils/

data  demo  evaluation	models	run  utils
load.py  modules  outfit_clip_transformer.py  outfit_transformer.py
collate_fn.py  datasets  datatypes.py
1_generate_rec_embeddings.py  3_run.py	      vectorstore_utils.py
2_build_index.py	      vectorstore.py
metrics.py
1_generate_clip_embeddings.py  3_test_complementary.py
2_test_compatibility.py        3_train_complementary.py
2_train_compatibility.py
distributed_utils.py  logger.py  loss.py  model_utils.py  utils.py


In [ ]:
!cat src/models/outfit_transformer.py
!cat src/data/datatypes.py

In [ ]:
!ls src/models/modules/
!cat src/models/modules/*.py   # show all files inside

In [ ]:
!ls src/data/datasets/

In [ ]:
!cat src/run/2_train_compatibility.py
!cat src/data/datasets/*.py   # or ls src/data/datasets/ first

In [ ]:
!grep -rn "precompute_clip_embedding" src/

In [ ]:
!cat src/models/outfit_clip_transformer.py

In [ ]:
%%writefile src/models/outfit_transformer.py
from torch import nn
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Any, Union, Literal, Optional
from torch import Tensor
from PIL import Image
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import os
import pathlib
from ..data.datatypes import (
    FashionCompatibilityQuery, FashionComplementaryQuery, FashionItem
)
from .modules.encoder import ItemEncoder
from ..utils.model_utils import get_device

@dataclass
class OutfitTransformerConfig:
    padding: Literal['longest', 'max_length'] = 'longest'
    max_length: int = 16
    truncation: bool = True

    item_enc_text_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    item_enc_dim_per_modality: int = 128
    item_enc_norm_out: bool = True
    aggregation_method: Literal['concat', 'sum', 'mean'] = 'concat'

    transformer_n_head: int = 16 # Original: 16
    transformer_d_ffn: int = 2024 # Original: Unknown
    transformer_n_layers: int = 6 # Original: 6
    transformer_dropout: float = 0.3 # Original: Unknown
    transformer_norm_out: bool = False

    d_embed: int = 128

    image_only: bool = False  # ← ADD THIS LINE


class OutfitTransformer(nn.Module):

    def __init__(self, cfg: Optional[OutfitTransformerConfig] = None):
        super().__init__()
        self.cfg = cfg if cfg is not None else OutfitTransformerConfig()
        self._init_item_enc()
        self._init_style_enc()
        self._init_variables()

    def _init_item_enc(self):
        """Builds the outfit encoder using configuration parameters."""
        self.item_enc = ItemEncoder(
            text_model_name=self.cfg.item_enc_text_model_name,
            enc_dim_per_modality=self.cfg.item_enc_dim_per_modality,
            enc_norm_out=self.cfg.item_enc_norm_out,
            aggregation_method=self.cfg.aggregation_method,
            image_only=self.cfg.image_only
        )

    def _init_style_enc(self):
        """Builds the transformer encoder using configuration parameters."""
        style_enc_layer = nn.TransformerEncoderLayer(
            d_model=self.item_enc.d_embed,
            nhead=self.cfg.transformer_n_head,
            dim_feedforward=self.cfg.transformer_d_ffn,
            dropout=self.cfg.transformer_dropout,
            batch_first=True,
            norm_first=True,
            activation=F.mish,
        )
        self.style_enc = nn.TransformerEncoder(
            encoder_layer=style_enc_layer,
            num_layers=self.cfg.transformer_n_layers,
            enable_nested_tensor=False
        )
        self.predict_ffn = nn.Sequential(
            # nn.LayerNorm(self.item_enc.d_embed),
            nn.Dropout(self.cfg.transformer_dropout),
            nn.Linear(self.item_enc.d_embed, 1),
            nn.Sigmoid()
        )
        self.embed_ffn = nn.Sequential(
            nn.Linear(self.item_enc.d_embed, self.cfg.d_embed, bias=False)
        )

    def _init_variables(self):
        image_size = (self.item_enc.image_size, self.item_enc.image_size)
        # self.image_query = Image.open(self.cfg.query_img_path).resize(image_size)
        self.image_pad = Image.new("RGB", image_size)
        self.text_pad = ''

        self.task_emb = nn.Parameter(
            torch.randn(self.item_enc.d_embed // 2) * 0.02, requires_grad=True
        )
        self.predict_emb = nn.Parameter(
            torch.randn(self.item_enc.d_embed // 2) * 0.02, requires_grad=True
        )
        self.embed_emb = nn.Parameter(
            torch.randn(self.item_enc.d_embed // 2) * 0.02, requires_grad=True
        )
        self.pad_emb = nn.Parameter(
            torch.randn(self.item_enc.d_embed) * 0.02, requires_grad=True
        )

    def _get_max_length(self, sequences):
        if self.cfg.padding == 'max_length':
            return self.cfg.max_length
        max_length = max(len(seq) for seq in sequences)

        return min(self.cfg.max_length, max_length) if self.cfg.truncation else max_length

    def _pad_sequences(self, sequences, pad_value, max_length):
        return [seq[:max_length] + [pad_value] * (max_length - len(seq)) for seq in sequences]

    def _pad_and_mask_for_outfits(self, outfits):
        max_length = self._get_max_length(outfits)
        images = self._pad_sequences(
            [[item.image for item in outfit] for outfit in outfits],
            self.image_pad, max_length
        )
        texts = self._pad_sequences(
            [[f"{item.description}" for item in outfit] for outfit in outfits],
            self.text_pad, max_length
        )
        mask = [[0] * len(seq) + [1] * (max_length - len(seq)) for seq in outfits]

        return images, texts, torch.BoolTensor(mask).to(self.device)

    def _pad_and_mask_for_embs(self, embs_of_outfits):
        max_length = self._get_max_length(embs_of_outfits)
        batch_size = len(embs_of_outfits)

        embeddings = torch.empty((batch_size, max_length, self.item_enc.d_embed),
                                 dtype=torch.float, device=self.device)
        mask = []

        for i, embs_of_outfit in enumerate(embs_of_outfits):
            embs_of_outfit = torch.tensor(
                np.array(embs_of_outfit[:max_length]), dtype=torch.float
            ).to(self.device)
            length = len(embs_of_outfit)

            embeddings[i, :length] = embs_of_outfit
            embeddings[i, length:] = self.pad_emb  # 패딩 부분을 학습 가능한 벡터로 채움
            mask.append([0] * length + [1] * (max_length - length))

        return embeddings, torch.BoolTensor(mask).to(self.device)

    def _style_enc_forward(self, embs_of_inputs, src_key_padding_mask):
        if self.cfg.aggregation_method == 'concat':
            half_d_embed = self.item_enc.d_embed // 2
            normalized_embs = torch.cat([
                F.normalize(embs_of_inputs[:, :, :half_d_embed], p=2, dim=-1),
                F.normalize(embs_of_inputs[:, :, half_d_embed:], p=2, dim=-1)
            ], dim=-1)
        else:
            normalized_embs = F.normalize(embs_of_inputs, p=2, dim=-1)

        return self.style_enc(normalized_embs, src_key_padding_mask=src_key_padding_mask)

    def predict_score(self, query: List[FashionCompatibilityQuery], use_precomputed_embedding: bool = False) -> Tensor:
        outfits = [query_.outfit for query_ in query]
        if use_precomputed_embedding:
            assert all([item_.embedding is not None for item_ in sum(outfits, [])])
            embs_of_inputs = [[item_.embedding for item_ in outfit] for outfit in outfits]
            embs_of_inputs, mask = self._pad_and_mask_for_embs(embs_of_inputs)
        else:
            outfits = [query_.outfit for query_ in query]
            images, texts, mask = self._pad_and_mask_for_outfits(outfits)
            embs_of_inputs = self.item_enc(images, texts)

        task_emb = torch.cat([self.task_emb, self.predict_emb], dim=-1)

        embs_of_inputs = torch.cat([
            task_emb.view(1, 1, -1).expand(len(query), -1, -1), # [B, 1, D]
            embs_of_inputs # [B, L, D]
        ], dim=1) # [B, L+1, D]
        mask = torch.cat([
            torch.zeros(len(query), 1, dtype=torch.bool, device=self.device), # [B, 1]
            mask # [B, L]
        ], dim=1) # [B, L+1]

        last_hidden_states = self._style_enc_forward(embs_of_inputs, src_key_padding_mask=mask)
        scores = self.predict_ffn(last_hidden_states[:, 0, :])

        return scores

    def embed_query(self, query: List[FashionComplementaryQuery], use_precomputed_embedding: bool=False) -> Tensor:
        # q_items = [[FashionItem(category=i.category, image=self.image_query, description=i.category)] for i in query]
        outfits = [query_.outfit for query_ in query]
        if use_precomputed_embedding:
            assert all([item_.embedding is not None for item_ in sum(outfits, [])])
            embs_of_inputs = [[item_.embedding for item_ in outfit] for outfit in outfits]
            embs_of_inputs, mask = self._pad_and_mask_for_embs(embs_of_inputs)
        else:
            images, texts, mask = self._pad_and_mask_for_outfits(outfits)
            embs_of_inputs = self.item_enc(images, texts)

        task_emb = torch.cat([self.task_emb, self.embed_emb], dim=-1)

        embs_of_inputs = torch.cat([
            task_emb.view(1, 1, -1).expand(len(query), -1, -1), embs_of_inputs
        ], dim=1)
        mask = torch.cat([
            torch.zeros(len(query), 1, dtype=torch.bool, device=self.device), mask
        ], dim=1)

        last_hidden_states = self._style_enc_forward(embs_of_inputs, src_key_padding_mask=mask)
        embeddings = self.embed_ffn(last_hidden_states[:, 0, :])

        return F.normalize(embeddings, p=2, dim=-1) if self.cfg.transformer_norm_out else embeddings

    def embed_item(self, item: List[FashionItem], use_precomputed_embedding: bool=False) -> Tensor:
        if use_precomputed_embedding:
            assert all([item_.embedding is not None for item_ in item])
            embs_of_inputs = [[item_.embedding] for item_ in item]
            embs_of_inputs, mask = self._pad_and_mask_for_embs(embs_of_inputs)
        else:
            outfits = [[item_] for item_ in item]
            images, texts, mask = self._pad_and_mask_for_outfits(outfits)
            embs_of_inputs = self.item_enc(images, texts)

        last_hidden_states = self._style_enc_forward(embs_of_inputs, src_key_padding_mask=mask)
        embeddings = self.embed_ffn(last_hidden_states[:, 0, :]) # [B, D]

        return F.normalize(embeddings, p=2, dim=-1) if self.cfg.transformer_norm_out else embeddings

    def forward(
        self,
        inputs: List[Union[FashionCompatibilityQuery, FashionComplementaryQuery, FashionItem]],
        *args, **kwargs
    ) -> Tensor:
        if isinstance(inputs[0], FashionCompatibilityQuery):
            return self.predict_score(inputs, *args, **kwargs)

        elif isinstance(inputs[0], FashionComplementaryQuery):
            return self.embed_query(inputs, *args, **kwargs)

        elif isinstance(inputs[0], FashionItem):
            return self.embed_item(inputs, *args, **kwargs)
        else:
            raise ValueError("Invalid input type.")

    @property
    def device(self) -> torch.device:
        """Returns the device on which the model's parameters are stored."""
        return get_device(self)

In [ ]:
%%writefile src/models/outfit_clip_transformer.py
import torch
from torch import nn
from typing import List, Tuple, Union
from ..data.datatypes import FashionItem
from dataclasses import dataclass
from .modules.encoder import CLIPItemEncoder
from .outfit_transformer import OutfitTransformer, OutfitTransformerConfig
import numpy as np

@dataclass
class OutfitCLIPTransformerConfig(OutfitTransformerConfig):
    item_enc_clip_model_name: str = "patrickjohncyh/fashion-clip"


class OutfitCLIPTransformer(OutfitTransformer):

    def __init__(
        self,
        cfg: OutfitCLIPTransformerConfig = OutfitCLIPTransformerConfig()
    ):
        super().__init__(cfg)

    def _init_item_enc(self) -> CLIPItemEncoder:
        self.item_enc = CLIPItemEncoder(
            model_name=self.cfg.item_enc_clip_model_name,
            enc_norm_out=self.cfg.item_enc_norm_out,
            aggregation_method=self.cfg.aggregation_method,
            image_only=self.cfg.image_only
        )

    def precompute_clip_embedding(self, item: List[FashionItem]) -> np.ndarray:
        outfits = [[item_] for item_ in item]
        images, texts, mask = self._pad_and_mask_for_outfits(outfits)
        enc_outs = self.item_enc(images, texts)
        embeddings = enc_outs[:, 0, :]
        return embeddings.detach().cpu().numpy()

In [ ]:
%%writefile src/models/modules/encoder.py
from torch import nn
from dataclasses import dataclass, field

from typing import List, Tuple, Dict, Any, Union, Literal, Optional
from torch import Tensor
from PIL import Image
import numpy as np
import torch
import torch.nn.functional as F
import os

# Import custom modules
from ...data import datatypes
from .image_encoder import Resnet18ImageEncoder, CLIPImageEncoder
from .text_encoder import HuggingFaceTextEncoder, CLIPTextEncoder
from ...utils.model_utils import freeze_model, mean_pooling, aggregate_embeddings
from transformers import AutoModel, AutoTokenizer, AutoProcessor


class ItemEncoder(nn.Module):
    def __init__(
        self,
        model_name,
        enc_dim_per_modality,
        enc_norm_out,
        aggregation_method,
        image_only: bool = False      # ← ADD THIS
    ):
        super().__init__()
        self.enc_dim_per_modality = enc_dim_per_modality
        self.aggregation_method = aggregation_method
        self.enc_norm_out = enc_norm_out
        self.image_only = image_only  # ← ADD THIS
        self._build_encoders(model_name)

        if image_only:                # ← ADD THIS BLOCK
            self.learned_no_text_emb = nn.Parameter(
                torch.randn(enc_dim_per_modality) * 0.02, requires_grad=True
            )

    def forward(self, images, texts, *args, **kwargs):
        image_embeddings = self.image_enc(
            images, normalize=self.enc_norm_out, *args, **kwargs
        )

        if self.image_only:           # ← ADD THIS BLOCK
            B, L, _ = image_embeddings.shape
            text_embeddings = self.learned_no_text_emb\
                .unsqueeze(0).unsqueeze(0)\
                .expand(B, L, -1)
        else:
            text_embeddings = self.text_enc(
                texts, normalize=self.enc_norm_out, *args, **kwargs
            )

        encoder_outputs = aggregate_embeddings(
            image_embeddings=image_embeddings,
            text_embeddings=text_embeddings,
            aggregation_method=self.aggregation_method
        )
        return encoder_outputs


class CLIPItemEncoder(ItemEncoder):
    def __init__(
        self,
        model_name,
        enc_norm_out,
        aggregation_method,
	image_only: bool = False  # ← ADD
    ):
        super().__init__(
            model_name=model_name,
            enc_dim_per_modality=512,
            enc_norm_out=enc_norm_out,
            aggregation_method=aggregation_method,
	    image_only=image_only  # ← ADD
        )

    def _build_encoders(self, model_name):
        self.image_enc = CLIPImageEncoder(
            model_name_or_path=model_name
        )
        self.text_enc = CLIPTextEncoder(
            model_name_or_path=model_name
        )

In [ ]:
%%writefile src/run/1_generate_clip_embeddings.py
import json
import logging
import os
import pathlib
import pickle
import sys
import tempfile
from argparse import ArgumentParser
from typing import Any, Dict, List, Literal, Optional

import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
from tqdm import tqdm

import wandb

from ..data import collate_fn
from ..data.datasets import polyvore
from ..models.load import load_model
from ..utils.distributed_utils import cleanup, setup
from ..utils.logger import get_logger
from ..utils.utils import seed_everything

SRC_DIR = pathlib.Path(__file__).parent.parent.parent.absolute()
LOGS_DIR = SRC_DIR / 'logs'
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs(LOGS_DIR, exist_ok=True)

POLYVORE_PRECOMPUTED_CLIP_EMBEDDING_DIR = "{polyvore_dir}/precomputed_clip_embeddings"


def parse_args():
    parser = ArgumentParser()
    parser.add_argument('--model_type', type=str, choices=['original', 'clip'],
                        default='clip')
    parser.add_argument('--polyvore_dir', type=str,
                        default='./datasets/polyvore')
    parser.add_argument('--polyvore_type', type=str, choices=['nondisjoint', 'disjoint'],
                        default='nondisjoint')
    parser.add_argument('--batch_sz_per_gpu', type=int,
                        default=128)
    parser.add_argument('--n_workers_per_gpu', type=int,
                        default=4)
    parser.add_argument('--checkpoint', type=str,
                        default=None)
    parser.add_argument('--world_size', type=int,
                        default=-1)
    parser.add_argument('--demo', action='store_true')
    parser.add_argument('--image_only', action='store_true', default=False)  # ← ADD
    parser.add_argument('--output_suffix', type=str, default='')             # ← ADD (for naming)

    return parser.parse_args()


def setup_dataloaders(rank, world_size, args):
    item_dataset = polyvore.PolyvoreItemDataset(
        dataset_dir=args.polyvore_dir, load_image=True
    )

    n_items = len(item_dataset)
    n_items_per_gpu = n_items // world_size

    start_idx = n_items_per_gpu * rank
    end_idx = (start_idx + n_items_per_gpu) if rank < world_size - 1 else n_items
    item_dataset = torch.utils.data.Subset(item_dataset, range(start_idx, end_idx))

    item_dataloader = DataLoader(
        dataset=item_dataset, batch_size=args.batch_sz_per_gpu, shuffle=False,
        num_workers=args.n_workers_per_gpu, collate_fn=collate_fn.item_collate_fn
    )

    return item_dataloader


def compute(rank: int, world_size: int, args: Any):
    # Setup
    setup(rank, world_size)

    # Logging Setup
    logger = get_logger('precompute_clip_embedding', LOGS_DIR, rank)
    logger.info(f'Logger Setup Completed')

    # Dataloaders
    item_dataloader = setup_dataloaders(rank, world_size, args)
    logger.info(f'Dataloaders Setup Completed')

    # Model setting
    model = load_model(
    	model_type=args.model_type,
    	checkpoint=args.checkpoint,
    	image_only=args.image_only  # ← ADD
	)
    model.eval()
    logger.info(f'Model Loaded')

    all_ids, all_embeddings = [], []
    with torch.no_grad():
        for batch in tqdm(item_dataloader):
            if args.demo and len(all_embeddings) > 10:
                break

            if dist.get_world_size() > 1:
                embeddings = model.module.precompute_clip_embedding(batch)  # (batch_size, d_embed)
            else:
                embeddings = model.precompute_clip_embedding(batch)

            all_ids.extend([item.item_id for item in batch])
            all_embeddings.append(embeddings)

    all_embeddings = np.concatenate(all_embeddings, axis=0)
    logger.info(f"Computed {len(all_embeddings)} embeddings")

    # numpy 어레이 저장
    suffix = '_image_only' if args.image_only else ''
    save_dir = POLYVORE_PRECOMPUTED_CLIP_EMBEDDING_DIR.format(polyvore_dir=args.polyvore_dir) + suffix
    os.makedirs(save_dir, exist_ok=True)
    save_path = f"{save_dir}/polyvore_{rank}.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump({'ids': all_ids, 'embeddings': all_embeddings}, f)

    # DDP 종료
    cleanup()


if __name__ == '__main__':
    args = parse_args()

    if args.world_size == -1:
        args.world_size = torch.cuda.device_count()

    mp.spawn(
        compute, args=(args.world_size, args),
        nprocs=args.world_size, join=True
    )

In [ ]:
%%writefile src/models/load.py
import torch
from typing import Any, Dict, Optional
from .outfit_transformer import (
    OutfitTransformerConfig,
    OutfitTransformer
)
from .outfit_clip_transformer import (
    OutfitCLIPTransformerConfig,
    OutfitCLIPTransformer
)
from torch.distributed import get_rank, get_world_size
from torch.nn.parallel import DistributedDataParallel as DDP


def load_model(model_type, checkpoint=None, image_only=False, **cfg_kwargs):
    is_distributed = torch.distributed.is_initialized()

    if is_distributed:
        rank = get_rank()
        world_size = get_world_size()
        map_location = f'cuda:{rank}' if torch.cuda.is_available() else 'cpu'
    else:
        rank = 0
        world_size = 1
        map_location = 'cuda' if torch.cuda.is_available() else 'cpu'

    state_dict = None
    if checkpoint:
        state_dict = torch.load(checkpoint, map_location=map_location)
        cfg = state_dict.get('config', {})
        cfg['image_only'] = image_only
        model_state_dict = state_dict.get('model', {})
    else:
        cfg = cfg_kwargs
        cfg['image_only'] = image_only
        model_state_dict = None

    if model_type == 'original':
        model = OutfitTransformer(OutfitTransformerConfig(**cfg))
    elif model_type == 'clip':
        model = OutfitCLIPTransformer(OutfitCLIPTransformerConfig(**cfg))
    else:
        raise ValueError(f"Unsupported model_type: {model_type}")

    model.to(rank)

    if model_state_dict:
        new_state_dict = {}
        for k, v in model_state_dict.items():
            new_key = k.replace("module.", "")
            new_state_dict[new_key] = v
        missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
        if missing:
            print(f"[Warning] Missing keys in state_dict: {missing}")
        if unexpected:
            print(f"[Warning] Unexpected keys in state_dict: {unexpected}")
        print(f"Loaded model from checkpoint: {checkpoint}")

    if world_size > 1:
        model = DDP(model, device_ids=[rank], static_graph=True)

    return model

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

In [ ]:
#!python -m src.run.2_train_compatibility \
#  --model_type clip \
#  --checkpoint checkpoints/compatibillity_clip_best.pth \
#  --n_epochs 10 \
#  --lr 1e-3 \
#  --world_size 1

#!python -m src.run.3_train_complementary \
#  --model_type clip \
#  --checkpoint checkpoints/complementary_clip_best.pth \
#  --n_epochs 10 \
#  --lr 1e-3 \
#  --world_size 1

In [ ]:
#!python -m src.run.1_generate_clip_embeddings \
#  --model_type clip \
#  --checkpoint checkpoints/compatibillity_clip_best.pth \
#  --image_only \
#  --polyvore_dir ./datasets/polyvore \
#  --world_size 1

In [ ]:
#!cat src/utils/distributed_utils.py

In [ ]:
content = '''import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP


def setup(rank, world_size):
    if world_size <= 1:
        return  # skip distributed setup for single process / CPU
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    dist.init_process_group(backend="nccl", rank=rank, world_size=world_size)


def cleanup():
    if dist.is_initialized():
        dist.destroy_process_group()


def gather_results(all_loss, all_preds, all_labels):
    if not dist.is_initialized() or dist.get_world_size() == 1:
        return all_loss, all_preds, all_labels

    gathered_preds = [torch.empty_like(all_preds) for _ in range(dist.get_world_size())]
    gathered_labels = [torch.empty_like(all_labels) for _ in range(dist.get_world_size())]
    dist.all_gather(gathered_preds, all_preds)
    dist.all_gather(gathered_labels, all_labels)
    dist.all_reduce(all_loss, op=dist.ReduceOp.SUM)
    all_loss /= dist.get_world_size()
    gathered_preds = torch.cat(gathered_preds, dim=0)
    gathered_labels = torch.cat(gathered_labels, dim=0)

    return all_loss, gathered_preds, gathered_labels
'''

with open('src/utils/distributed_utils.py', 'w') as f:
    f.write(content)

print("Done")

In [ ]:
#!tail -20 src/run/2_train_compatibility.py

In [ ]:
# 1. Re-apply all code changes (load.py, encoder.py, etc.)
#    since Colab resets the filesystem on runtime change

# 2. Generate image-only embeddings
!python -m src.run.1_generate_clip_embeddings \
  --model_type clip \
  --checkpoint checkpoints/compatibillity_clip_best.pth \
  --image_only \
  --polyvore_dir ./datasets/polyvore

# 3. Fine-tune compatibility
!python -m src.run.2_train_compatibility \
  --model_type clip \
  --checkpoint checkpoints/compatibillity_clip_best.pth \
  --n_epochs 10 \
  --lr 1e-3

# 4. Fine-tune complementary
!python -m src.run.3_train_complementary \
  --model_type clip \
  --checkpoint checkpoints/complementary_clip_best.pth \
  --n_epochs 10 \
  --lr 1e-3

README.md: 1.23kB [00:00, 1.59MB/s]
data/data-00000-of-00005.parquet: 100% 369M/369M [00:08<00:00, 44.8MB/s]
data/data-00001-of-00005.parquet: 100% 368M/368M [00:14<00:00, 25.1MB/s]
data/data-00002-of-00005.parquet: 100% 369M/369M [00:18<00:00, 19.6MB/s]
data/data-00003-of-00005.parquet: 100% 368M/368M [00:10<00:00, 35.4MB/s]
data/data-00004-of-00005.parquet: 100% 368M/368M [00:14<00:00, 25.2MB/s]
data/data-00005-of-00005.parquet: 100% 367M/367M [00:14<00:00, 24.8MB/s]
Generating data split: 251008 examples [01:04, 3904.08 examples/s]
251008it [02:48, 1487.73it/s]
251008it [00:00, 2091199.35it/s]
251008it [02:49, 1482.88it/s]
251008it [00:00, 1470550.49it/s]
2026-05-05 16:36:23,465 - INFO - [Rank 0] - Logger Setup Completed
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/outfit-transformer/src/run/1_generate_clip_embeddings.py", line 138, in <module>
    mp.spawn(
  File "/usr/l

In [ ]:
# Read the file
with open('src/run/1_generate_clip_embeddings.py', 'r') as f:
    content = f.read()

# Fix the outdated call
content = content.replace(
    'item_dataset = polyvore.PolyvoreItemDataset(\n        dataset_dir=args.polyvore_dir, load_image=True\n    )',
    'item_dataset = polyvore.PolyvoreItemDataset()'
)

with open('src/run/1_generate_clip_embeddings.py', 'w') as f:
    f.write(content)

print("Done")

Done


In [ ]:
!grep -n "PolyvoreItemDataset" src/run/1_generate_clip_embeddings.py

63:    item_dataset = polyvore.PolyvoreItemDataset()


In [ ]:
with open('src/models/modules/encoder.py', 'r') as f:
    content = f.read()

# Fix: rename the instance variable to avoid shadowing the property
content = content.replace(
    '''    def __init__(
        self, d_embed: int = 64,
        size: int = 224, crop_size: int = 224, freeze: bool = False
    ):
        super().__init__()

        # Load pre-trained ResNet-18 and adjust the final layer to match d_embed
        self.d_embed = d_embed''',
    '''    def __init__(
        self, d_embed: int = 64,
        size: int = 224, crop_size: int = 224, freeze: bool = False
    ):
        super().__init__()

        # Load pre-trained ResNet-18 and adjust the final layer to match d_embed
        self._d_embed = d_embed'''
)

content = content.replace(
    '''    @property
    def d_embed(self) -> int:
        return self.d_embed''',
    '''    @property
    def d_embed(self) -> int:
        return self._d_embed'''
)

with open('src/models/modules/encoder.py', 'w') as f:
    f.write(content)

print("Done")

Done


In [ ]:
content = '''from torch import nn
from typing import List, Dict, Any
import numpy as np
import torch
import torch.nn.functional as F

from ...data import datatypes
from .image_encoder import Resnet18ImageEncoder, CLIPImageEncoder
from .text_encoder import HuggingFaceTextEncoder, CLIPTextEncoder
from ...utils.model_utils import freeze_model, mean_pooling, aggregate_embeddings
from transformers import AutoModel, AutoTokenizer, AutoProcessor


class ItemEncoder(nn.Module):
    def __init__(
        self,
        model_name,
        enc_dim_per_modality,
        enc_norm_out,
        aggregation_method,
        image_only: bool = False
    ):
        super().__init__()
        self.enc_dim_per_modality = enc_dim_per_modality
        self.aggregation_method = aggregation_method
        self.enc_norm_out = enc_norm_out
        self.image_only = image_only
        self._build_encoders(model_name)

        if image_only:
            self.learned_no_text_emb = nn.Parameter(
                torch.randn(enc_dim_per_modality) * 0.02, requires_grad=True
            )

    def _build_encoders(self, model_name):
        self.image_enc = Resnet18ImageEncoder(
            embedding_size=self.enc_dim_per_modality,
        )
        self.text_enc = HuggingFaceTextEncoder(
            embedding_size=self.enc_dim_per_modality,
            model_name_or_path=model_name
        )

    @property
    def d_embed(self):
        if self.aggregation_method == "concat":
            return self.enc_dim_per_modality * 2
        else:
            return self.enc_dim_per_modality

    @property
    def image_size(self):
        return self.image_enc.image_size

    def forward(self, images, texts, *args, **kwargs):
        image_embeddings = self.image_enc(
            images, normalize=self.enc_norm_out, *args, **kwargs
        )

        if self.image_only:
            B, L, _ = image_embeddings.shape
            text_embeddings = self.learned_no_text_emb\\
                .unsqueeze(0).unsqueeze(0)\\
                .expand(B, L, -1)
        else:
            text_embeddings = self.text_enc(
                texts, normalize=self.enc_norm_out, *args, **kwargs
            )

        encoder_outputs = aggregate_embeddings(
            image_embeddings=image_embeddings,
            text_embeddings=text_embeddings,
            aggregation_method=self.aggregation_method
        )
        return encoder_outputs


class CLIPItemEncoder(ItemEncoder):
    def __init__(
        self,
        model_name,
        enc_norm_out,
        aggregation_method,
        image_only: bool = False
    ):
        super().__init__(
            model_name=model_name,
            enc_dim_per_modality=512,
            enc_norm_out=enc_norm_out,
            aggregation_method=aggregation_method,
            image_only=image_only
        )

    def _build_encoders(self, model_name):
        self.image_enc = CLIPImageEncoder(
            model_name_or_path=model_name
        )
        self.text_enc = CLIPTextEncoder(
            model_name_or_path=model_name
        )
'''

with open('src/models/modules/encoder.py', 'w') as f:
    f.write(content)

print("Done")

Done


In [ ]:
with open('src/data/datasets/polyvore.py', 'r') as f:
    content = f.read()

# Fix the index mapping - it should use the actual keys not sequential indices
content = content.replace(
    'metadata_idx2item_id = {\n    j: i for i, j in tqdm(enumerate(item_id2metadata_idx))\n}',
    'metadata_idx2item_id = {\n    i: j for i, j in enumerate(item_id2metadata_idx.keys())\n}'
)

with open('src/data/datasets/polyvore.py', 'w') as f:
    f.write(content)

print("Done")

Done


In [ ]:
!grep -n "metadata_idx2item_id" src/data/datasets/polyvore.py

38:metadata_idx2item_id = {
202:        item_id = metadata_idx2item_id[idx]


In [ ]:
with open('src/run/1_generate_clip_embeddings.py', 'r') as f:
    content = f.read()

content = content.replace(
    'if dist.get_world_size() > 1:\n                embeddings = model.module.precompute_clip_embedding(batch)  # (batch_size, d_embed)\n            else:\n                embeddings = model.precompute_clip_embedding(batch)',
    'if dist.is_initialized() and dist.get_world_size() > 1:\n                embeddings = model.module.precompute_clip_embedding(batch)  # (batch_size, d_embed)\n            else:\n                embeddings = model.precompute_clip_embedding(batch)'
)

with open('src/run/1_generate_clip_embeddings.py', 'w') as f:
    f.write(content)

print("Done")

Done


In [ ]:
!grep -n "dist.get_world_size\|dist.is_initialized" src/run/1_generate_clip_embeddings.py

107:            if dist.is_initialized() and dist.get_world_size() > 1:


In [ ]:
!python -m src.run.1_generate_clip_embeddings \
  --model_type clip \
  --checkpoint checkpoints/compatibillity_clip_best.pth \
  --image_only \
  --polyvore_dir ./datasets/polyvore

251008it [02:49, 1485.21it/s]
251008it [02:51, 1464.93it/s]
2026-05-06 15:58:23,186 - INFO - [Rank 0] - Logger Setup Completed
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-05-06 15:58:23,187 - INFO - [Rank 0] - Dataloaders Setup Completed
config.json: 4.46kB [00:00, 10.6MB/s]
model.safetensors: 100% 605M/605M [00:18<00:00, 33.2MB/s]
Loading weights: 100% 200/200 [00:00<00:00, 871.65it/s, Materializing param=visual_projection.weight]
CLIPVisionModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                             

In [ ]:
!ls -lh datasets/polyvore/precomputed_clip_embeddings_image_only/

total 982M
-rw-r--r-- 1 root root 982M May  6 16:43 polyvore_0.pkl


In [ ]:
# Point training scripts to image-only embeddings
for filepath in ['src/run/2_train_compatibility.py', 'src/run/3_train_complementary.py']:
    with open(filepath, 'r') as f:
        content = f.read()
    content = content.replace(
        'embedding_dict = polyvore.load_embedding_dict(args.polyvore_dir)',
        'embedding_dict = polyvore.load_embedding_dict(args.polyvore_dir + "/precomputed_clip_embeddings_image_only")'
    )
    # Also freeze everything except learned_no_text_emb
    content = content.replace(
        'model = load_model(model_type=args.model_type, checkpoint=args.checkpoint)',
        '''model = load_model(model_type=args.model_type, checkpoint=args.checkpoint, image_only=True)
    for name, param in model.named_parameters():
        param.requires_grad = ('learned_no_text_emb' in name)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable}")'''
    )
    with open(filepath, 'w') as f:
        f.write(content)
    print(f"Done: {filepath}")

Done: src/run/2_train_compatibility.py
Done: src/run/3_train_complementary.py


In [ ]:
# Step 2 - compatibility
!python -m src.run.2_train_compatibility \
  --model_type clip \
  --checkpoint checkpoints/compatibillity_clip_best.pth \
  --n_epochs 10 \
  --lr 1e-3

# Step 3 - complementary
!python -m src.run.3_train_complementary \
  --model_type clip \
  --checkpoint checkpoints/complementary_clip_best.pth \
  --n_epochs 10 \
  --lr 1e-3